# 4 · Die Kampagne: Pilot, Rollout, Schätzung
**Päckli & Co.** · Data Science für Marketing · FHNW CAS Data Driven Marketing

Jetzt entscheidet ihr: wen ihr anschreibt, was ihr empfehlt und ob ihr 10 % Rabatt gebt. Die Kunden sind simuliert,
aber die Spielregeln sind echt: Jede E-Mail kostet, Abmeldungen kosten, und der Rabatt gilt auch für Käufe, die
sowieso passiert wären.

**Spielregeln**
- **Pilot:** bis zu 600 Kunden, beliebige Aufteilung auf die Gruppen `control`, `email`, `email_discount`.
- E-Mails brauchen 1 bis 3 Produktempfehlungen (`rec_1`, `rec_2`, `rec_3`). Die Kontrollgruppe bekommt nichts.
- **Rollout:** nur Kunden, die im Pilot nicht dabei waren.
- Pro Runde genau **eine** Einreichung. `validate(...)` ist gratis und beliebig oft.
- Zahlen: Marge 40 %, Rabatt 10 % auf die Bestellung, E-Mail CHF 0.10. Wer sich abmeldet, ist für künftige E-Mails verloren.

**Zwei Preise**
1. Höchster **inkrementeller Gewinn** (Pilot + Rollout): Gewinn mit E-Mail minus Gewinn ohne E-Mail.
2. Ehrlichste **Schätzung** des eigenen Rollout-Effekts: zuerst zählen Intervalle, die die Wahrheit enthalten,
   je schmaler pro angeschriebenem Kunden, desto besser.

**Minimum:** Pilot und Rollout eingereicht, Schätzung mit 95-%-Intervall abgegeben.
**Erweiterung (optional):** E-Mail-Texte, Rabatt nur für bestimmte Kunden, ein Diagramm für die Geschäftsleitung.

**So arbeitet ihr in jedem Schritt**
1. Lest das **Ziel**.
2. Kopiert den **Prompt** ins Gemini-Fenster (Gemini-Symbol oben rechts) und lasst euch den Code erklären.
3. Fügt den Code in die leere Zelle ein und führt sie aus (▶ links oder `Shift + Enter`).
4. **Plausibel?** Prüft das Ergebnis mit gesundem Menschenverstand.
5. Steckt ihr fest, führt die **💡 Lösung** aus. Das ist keine Schande: Die Lösung liefert dieselben Tabellen wie euer Code.

In [ ]:
#@title ▶ Setup: tragt eure Teamnummer ein und führt diese Zelle zuerst aus
TEAM = 0  #@param {type:"integer"}
BASIS_URL = "https://raw.githubusercontent.com/badrustanicki/fhnw-ds-marketing-2026/main/"
import os, urllib.request
if not os.path.exists("kurs.py"):
    urllib.request.urlretrieve(BASIS_URL + "kurs.py", "kurs.py")
import duckdb, numpy as np, pandas as pd, matplotlib.pyplot as plt
from kurs import *
start(TEAM, BASIS_URL)

In [ ]:
#@title ▶ Aufholen: Ergebnisse aus Block 2 und 3 laden (eure eigenen, sonst das Musterergebnis)
churn_scores = aufholen("churn_scores")
empfehlungen = aufholen("empfehlungen")

## Teil A · Planen (14:40)

### 1 Lohnt sich der Rabatt? Break-even

**Ziel:** Von Hand rechnen, um wie viel die Kaufwahrscheinlichkeit mit Rabatt steigen muss, damit er sich lohnt.

**Prompt für Gemini** (kopieren und im Gemini-Fenster einfügen):
```text
Marge 40 %, Rabatt 10 % auf den Listenpreis. Um welchen Faktor muss die Kaufwahrscheinlichkeit mit Rabatt mindestens steigen, damit der erwartete Gewinn nicht sinkt? Rechne es in Python für eine treue Kundin (kauft sowieso mit 60 %) und einen gefährdeten Kunden (5 %) durch.
```

In [ ]:
# Euer Code: fügt hier ein, was Gemini vorschlägt, und führt die Zelle aus (▶).

In [ ]:
#@title 💡 Lösung: nur wenn ihr nicht weiterkommt (Doppelklick zeigt den Code, ▶ führt ihn aus)
marge, rabatt = 0.40, 0.10
faktor = marge / (marge - rabatt)
print(f"Die Kaufwahrscheinlichkeit muss um den Faktor {faktor:.2f} steigen (+{faktor - 1:.0%}).")
for name, p in [("Treue Kundin", 0.60), ("Gefährdeter Kunde", 0.05)]:
    print(f"{name}: von {p:.0%} auf mindestens {p * faktor:.1%}")

**Plausibel?** Die treue Kundin bräuchte 80 %: praktisch unmöglich. Der gefährdete Kunde braucht 6.7 %: gut machbar. Rabatt lohnt sich also, wenn überhaupt, bei Kunden, die ohne ihn kaum kaufen würden.

### 2 Die Zielgruppe

**Ziel:** Kunden finden, die gefährdet, aber noch erreichbar sind. Wer seit über einem Jahr still ist, liest keine E-Mails mehr; wer sowieso kauft, braucht keine.

**Prompt für Gemini** (kopieren und im Gemini-Fenster einfügen):
```text
Wähle aus churn_scores eine Zielgruppe: churn_risiko mindestens 0.5 und tage_seit_kauf höchstens 180. Speichere sie als DataFrame zielgruppe. Wie viele Kunden sind das? Zeichne ein Streudiagramm tage_seit_kauf gegen churn_risiko für alle Kunden und färbe die Zielgruppe ein.
```

In [ ]:
# Euer Code: fügt hier ein, was Gemini vorschlägt, und führt die Zelle aus (▶).

In [ ]:
#@title 💡 Lösung: nur wenn ihr nicht weiterkommt (Doppelklick zeigt den Code, ▶ führt ihn aus)
zielgruppe = churn_scores[(churn_scores["churn_risiko"] >= 0.5) & (churn_scores["tage_seit_kauf"] <= 180)]
print(f"Zielgruppe: {len(zielgruppe)} von {len(churn_scores)} Kunden")
ax = churn_scores.plot.scatter(x="tage_seit_kauf", y="churn_risiko", s=3, color="lightgray", figsize=(7, 4))
zielgruppe.plot.scatter(x="tage_seit_kauf", y="churn_risiko", s=3, ax=ax, title="Zielgruppe")
plt.show()

**Plausibel?** Die vier Kundentypen: Sichere Käufer (tiefes Risiko) bringen mit E-Mail nichts, Verlorene (lange still) sind kaum erreichbar. Wo vermutet ihr die Überzeugbaren? Ist die Zielgruppe gross genug für Pilot und Rollout?

### 3 Eure Pilotfrage
Bevor ihr Daten seht: Welche **eine** Frage soll der Pilot beantworten? Zum Beispiel: «Lohnt sich der Rabatt
bei unserer Zielgruppe?» Wer den Pilot sechsmal aufschneidet, findet immer einen «Effekt». Meist ist es Rauschen.

**Unsere Pilotfrage:** *(Doppelklick, hier eintragen)*

### E1 E-Mails schreiben lassen (optional)

**Ziel:** Sehen, wie Personalisierung im grossen Stil aussieht, und wo sie schiefgeht. Wird nicht bewertet.

**Prompt für Gemini** (kopieren und im Gemini-Fenster einfügen):
```text
Zeige für 3 Kunden aus zielgruppe die Produktnamen (description) ihrer Empfehlungen. Schreibe dann für jeden eine kurze, persönliche E-Mail von Päckli & Co. mit diesen drei Produkten.
```

In [ ]:
# Euer Code: fügt hier ein, was Gemini vorschlägt, und führt die Zelle aus (▶).

In [ ]:
#@title 💡 Lösung: nur wenn ihr nicht weiterkommt (Doppelklick zeigt den Code, ▶ führt ihn aus)
beispiel = zielgruppe.head(3).merge(empfehlungen, on="customer_id")
namen = duckdb.sql("SELECT stock_code, description FROM products").df().set_index("stock_code")["description"]
for _, k in beispiel.iterrows():
    print(k["customer_id"], "| zuletzt vor", k["tage_seit_kauf"], "Tagen |", ", ".join(namen[[k["rec_1"], k["rec_2"], k["rec_3"]]]))

**Plausibel?** Würdet ihr diese E-Mail selbst öffnen? Passen die Produkte zueinander? Erfindet die KI Rabatte oder Produkte, die es nicht gibt?

## Teil B · Pilot (ca. 15:30)

### 4 Die Pilot-Tabelle

**Ziel:** Zufällig Kunden aus der Zielgruppe ziehen und zufällig auf die drei Gruppen verteilen. Nur der Zufall macht die Gruppen vergleichbar.

**Prompt für Gemini** (kopieren und im Gemini-Fenster einfügen):
```text
Ziehe zufällig 600 Kunden aus zielgruppe (random_state=1) und teile sie zufällig in drei gleich grosse Gruppen control, email und email_discount (Spalte arm). Hänge rec_1, rec_2, rec_3 aus empfehlungen an; die Kontrollgruppe bekommt keine Empfehlungen. Ergebnis: DataFrame pilot mit customer_id, arm, rec_1, rec_2, rec_3. Prüfe ihn mit validate(pilot, 'pilot').
```

In [ ]:
# Euer Code: fügt hier ein, was Gemini vorschlägt, und führt die Zelle aus (▶).

In [ ]:
#@title 💡 Lösung: nur wenn ihr nicht weiterkommt (Doppelklick zeigt den Code, ▶ führt ihn aus)
n = min(600, len(zielgruppe) // 3 * 3)
pilot = zielgruppe.sample(n=n, random_state=1)[["customer_id"]].reset_index(drop=True)
pilot["arm"] = np.repeat(["control", "email", "email_discount"], n // 3)   # Reihenfolge ist schon zufällig
pilot = pilot.merge(empfehlungen[["customer_id", "rec_1", "rec_2", "rec_3"]], on="customer_id", how="left")
pilot.loc[pilot["arm"] == "control", ["rec_1", "rec_2", "rec_3"]] = None
validate(pilot, "pilot")

**Plausibel?** Zeigt validate ✅? Sind die Gruppen etwa gleich gross? Haben die E-Mail-Gruppen Empfehlungen, die Kontrollgruppe keine?

### 5 Pilot einreichen
Erst wenn `validate` ✅ zeigt. **Pro Runde nur einmal!** Colab fragt zur Sicherheit nach: «ja» eintippen.

In [ ]:
submit(pilot, "pilot")

### ⏸ Warten
Die Kursleitung wertet alle Piloten aus, sobald alle eingereicht haben. Danach liegt das Ergebnis in eurem Team-Ordner.
Kommt eine Meldung «noch nicht da», wartet kurz und führt die Zelle nochmals aus.

In [ ]:
pilot_ergebnis = load_results("pilot")
pilot_ergebnis.head()

### 6 Pilot auswerten

**Ziel:** Pro Gruppe sehen, was passiert ist. Eine Zeile pro Kunde: gekauft (purchased), Umsatz, Rabattkosten, Abmeldung (unsubscribed) und der realisierte Gewinn (profit).

**Prompt für Gemini** (kopieren und im Gemini-Fenster einfügen):
```text
Zeige für pilot_ergebnis pro arm: Anzahl Kunden, Kaufrate (Mittelwert von purchased), Ø Umsatz, Ø Gewinn (profit) und Anzahl Abmeldungen (unsubscribed).
```

In [ ]:
# Euer Code: fügt hier ein, was Gemini vorschlägt, und führt die Zelle aus (▶).

In [ ]:
#@title 💡 Lösung: nur wenn ihr nicht weiterkommt (Doppelklick zeigt den Code, ▶ führt ihn aus)
pilot_ergebnis.groupby("arm").agg(kunden=("customer_id", "size"), kaufrate=("purchased", "mean"),
                                  umsatz_avg=("revenue", "mean"), gewinn_avg=("profit", "mean"),
                                  abmeldungen=("unsubscribed", "sum")).round(2)

**Plausibel?** Die höchste Kaufrate heisst nicht der höchste Effekt: Was zählt, ist der Ø Gewinn **gegenüber der Kontrollgruppe**. Ist der Unterschied gross genug, dass er nicht Zufall sein kann? Das klärt der nächste Schritt.

### 7 Bootstrap: Wie sicher ist der Unterschied?

**Ziel:** Mit dem Bootstrap ein 95-%-Intervall für den Unterschied im Ø Gewinn berechnen: E-Mail gegen Kontrolle und E-Mail mit Rabatt gegen Kontrolle.

**Prompt für Gemini** (kopieren und im Gemini-Fenster einfügen):
```text
Bootstrap auf pilot_ergebnis: Ziehe 2000-mal aus jeder Gruppe (arm) mit Zurücklegen eine neue Stichprobe gleicher Grösse und berechne jedes Mal den Unterschied im Ø profit: email minus control und email_discount minus control. Gib pro Vergleich den Mittelwert und das 95-%-Intervall (2.5 %- und 97.5 %-Perzentil) aus und zeichne beide Verteilungen als Histogramm. Speichere die Mittelwerte im Dictionary effekt.
```

In [ ]:
# Euer Code: fügt hier ein, was Gemini vorschlägt, und führt die Zelle aus (▶).

In [ ]:
#@title 💡 Lösung: nur wenn ihr nicht weiterkommt (Doppelklick zeigt den Code, ▶ führt ihn aus)
rng = np.random.default_rng(0)
def bootstrap_diff(a, b, n=2000):
    """Unterschied der Mittelwerte a minus b, n-mal mit Zurücklegen neu gezogen."""
    a, b = np.asarray(a), np.asarray(b)
    return np.array([rng.choice(a, len(a)).mean() - rng.choice(b, len(b)).mean() for _ in range(n)])

gewinn = {arm: g["profit"].to_numpy() for arm, g in pilot_ergebnis.groupby("arm")}
effekt = {}
for arm in ["email", "email_discount"]:
    if arm in gewinn and "control" in gewinn:
        d = bootstrap_diff(gewinn[arm], gewinn["control"])
        effekt[arm] = d.mean()
        lo, hi = np.percentile(d, [2.5, 97.5])
        print(f"{arm} minus control: {d.mean():.2f} CHF pro Kunde, 95-%-Intervall {lo:.2f} bis {hi:.2f}")
        plt.hist(d, bins=40, alpha=0.6, label=arm)
plt.axvline(0, color="black"); plt.xlabel("Unterschied im Ø Gewinn pro Kunde (CHF)"); plt.legend(); plt.show()

**Plausibel?** Liegt 0 im Intervall? Dann wisst ihr noch nicht, ob es wirkt. Ein breites Intervall ist keine Schande, ein schmales um die falsche Zahl schon. Beantwortet jetzt eure Pilotfrage, und nur sie.

## Teil C · Rollout und Schätzung (ca. 16:00)

### 8 Die Rollout-Tabelle

**Ziel:** Entscheiden, wen ihr im Rollout anschreibt und mit welchem Angebot, und eine zufällige Kontrollgruppe abtrennen. Ohne sie könnt ihr den Effekt nicht schätzen. Eine grössere Kontrollgruppe gibt ein schmaleres Intervall, kostet aber Gewinn.

**Prompt für Gemini** (kopieren und im Gemini-Fenster einfügen):
```text
Erstelle die Rollout-Tabelle: alle Kunden aus zielgruppe, die nicht in pilot waren. Ziehe zufällig 20 % als Kontrollgruppe ab (arm = control, random_state=2); alle anderen bekommen den arm mit dem grösseren Effekt im Dictionary effekt. Empfehlungen wie im Pilot. Ergebnis: DataFrame rollout. Prüfe mit validate(rollout, 'rollout').
```

In [ ]:
# Euer Code: fügt hier ein, was Gemini vorschlägt, und führt die Zelle aus (▶).

In [ ]:
#@title 💡 Lösung: nur wenn ihr nicht weiterkommt (Doppelklick zeigt den Code, ▶ führt ihn aus)
arm_rollout = max(effekt, key=effekt.get) if effekt else "email"
rollout = zielgruppe[~zielgruppe["customer_id"].isin(pilot["customer_id"])][["customer_id"]]
rollout = rollout.sample(frac=1, random_state=2).reset_index(drop=True)   # zufällige Reihenfolge
n_kontrolle = round(0.2 * len(rollout))
rollout["arm"] = ["control"] * n_kontrolle + [arm_rollout] * (len(rollout) - n_kontrolle)
rollout = rollout.merge(empfehlungen[["customer_id", "rec_1", "rec_2", "rec_3"]], on="customer_id", how="left")
rollout.loc[rollout["arm"] == "control", ["rec_1", "rec_2", "rec_3"]] = None
print("Rollout mit", arm_rollout)
validate(rollout, "rollout")

**Plausibel?** Ist niemand aus dem Pilot dabei? Ist die Kontrollgruppe zufällig gezogen? Würdet ihr nach dem Pilot überhaupt anschreiben, und wen? Ihr dürft die Zielgruppe auch ändern.

### 9 Rollout einreichen
Erst wenn `validate` ✅ zeigt. **Nur einmal!**

In [ ]:
submit(rollout, "rollout")

### ⏸ Warten
Die Kursleitung wertet die Rollouts aus. Danach das Ergebnis laden.

In [ ]:
rollout_ergebnis = load_results("rollout")
rollout_ergebnis.head()

### 10 Die Schätzung für Preis 2

**Ziel:** Den inkrementellen Gewinn des Rollouts schätzen, mit 95-%-Intervall: (Ø Gewinn E-Mail − Ø Gewinn Kontrolle) × Anzahl angeschriebene Kunden.

**Prompt für Gemini** (kopieren und im Gemini-Fenster einfügen):
```text
Schätze den inkrementellen Gewinn des Rollouts aus rollout_ergebnis: für jeden arm ausser control (Ø profit dieses arm − Ø profit control) × Anzahl Kunden in diesem arm, summiert. Berechne mit dem Bootstrap (2000 Ziehungen, jede Gruppe mit Zurücklegen neu gezogen) ein 95-%-Intervall. Speichere schaetzung, ci_low, ci_high.
```

In [ ]:
# Euer Code: fügt hier ein, was Gemini vorschlägt, und führt die Zelle aus (▶).

In [ ]:
#@title 💡 Lösung: nur wenn ihr nicht weiterkommt (Doppelklick zeigt den Code, ▶ führt ihn aus)
gruppen = {arm: g["profit"].to_numpy() for arm, g in rollout_ergebnis.groupby("arm")}
kontrolle = gruppen["control"]
def inkrementell(gezogen):
    k = gezogen["control"].mean()
    return sum((gezogen[a].mean() - k) * len(gruppen[a]) for a in gruppen if a != "control")

schaetzung = inkrementell(gruppen)
boot = np.array([inkrementell({a: rng.choice(g, len(g)) for a, g in gruppen.items()}) for _ in range(2000)])
ci_low, ci_high = np.percentile(boot, [2.5, 97.5])
print(f"Schätzung: {schaetzung:.0f} CHF, 95-%-Intervall {ci_low:.0f} bis {ci_high:.0f} CHF")

**Plausibel?** Liegt die Schätzung im Intervall? Ist das Intervall so breit, dass es nichts aussagt? Ein Satz für die Geschäftsleitung: «Unsere Kampagne brachte zwischen X und Y CHF zusätzlichen Gewinn.»

### 11 Schätzung abgeben
Bis zur Auflösung dürft ihr sie noch ändern: einfach nochmals abgeben.

In [ ]:
submit_estimate(schaetzung, ci_low, ci_high)

## Erweiterung (optional)

### E2 Ein Diagramm, ein Satz

**Ziel:** Das Ergebnis für die Geschäftsleitung: ein Diagramm, ein Satz.

**Prompt für Gemini** (kopieren und im Gemini-Fenster einfügen):
```text
Zeichne ein einfaches Balkendiagramm mit dem geschätzten inkrementellen Gewinn des Rollouts und dem 95-%-Intervall als Fehlerbalken. Achsen beschriften, kein 3D. Schreibe darunter einen Satz für die Geschäftsleitung.
```

In [ ]:
# Euer Code: fügt hier ein, was Gemini vorschlägt, und führt die Zelle aus (▶).

In [ ]:
#@title 💡 Lösung: nur wenn ihr nicht weiterkommt (Doppelklick zeigt den Code, ▶ führt ihn aus)
plt.figure(figsize=(4, 3))
plt.bar(["Rollout"], [schaetzung], yerr=[[schaetzung - ci_low], [ci_high - schaetzung]], capsize=8)
plt.axhline(0, color="black"); plt.ylabel("Zusätzlicher Gewinn (CHF)")
plt.title("Inkrementeller Gewinn mit 95-%-Intervall"); plt.show()
print(f"Unsere Kampagne brachte zwischen {ci_low:.0f} und {ci_high:.0f} CHF zusätzlichen Gewinn.")

**Plausibel?** Versteht jemand ohne Statistik-Kenntnisse das Diagramm in zehn Sekunden?